In [ ]:

# ============================================================
# BƯỚC 1: DATA CLEANING — Top 200 Gia Vị Shopee
# Chạy trên Google Colab
# ============================================================

import pandas as pd
import numpy as np
import re

# ── 1. LOAD DATA ─────────────────────────────────────────────
df = pd.read_csv('/content/dataset.csv')
print("Shape gốc:", df.shape)
print(df.head(3))


# ── 2. ĐỔI TÊN CỘT CHO GỌN ──────────────────────────────────
df.columns = [
    'stt', 'ten_sp', 'gia_ban', 'doanh_so_365',
    'so_luong_ban_365', 'tong_doanh_so', 'tong_luot_ban',
    'tong_danh_gia', 'brand'
]


# ── 3. FIX CỘT GIÁ BÁN ──────────────────────────────────────
# Lỗi: một số ô bị dính ký tự kiểu "72.258.488₫14%" hoặc "68.479.200₫295%"
# Mục tiêu: chỉ lấy phần số, bỏ ₫ và % và mọi thứ phía sau

def clean_price(val):
    val = str(val)
    # Bỏ dấu chấm phân cách hàng nghìn kiểu VN, giữ lại chữ số
    # Lấy phần số đầu tiên trước ký tự ₫ hoặc chữ cái
    match = re.match(r'^[\d\.]+', val.replace(',', '.'))
    if match:
        num_str = match.group().replace('.', '')  # bỏ dấu chấm nghìn
        return int(num_str) if num_str else np.nan
    return np.nan

df['gia_ban'] = df['gia_ban'].apply(clean_price)
print("\n✅ Giá bán sau khi clean (5 dòng đầu):")
print(df['gia_ban'].head())


# ── 4. FIX CÁC CỘT SỐ KHÁC ──────────────────────────────────
# Các cột này cũng dùng dấu chấm kiểu VN (1.000 = một nghìn)
def clean_number(val):
    val = str(val).strip()
    val = re.sub(r'[^\d]', '', val)  # chỉ giữ chữ số
    return int(val) if val else np.nan

for col in ['doanh_so_365', 'so_luong_ban_365', 'tong_doanh_so',
            'tong_luot_ban', 'tong_danh_gia']:
    df[col] = df[col].apply(clean_number)

print("\n✅ Các cột số sau khi clean:")
print(df[['doanh_so_365', 'tong_luot_ban', 'tong_danh_gia']].head())


# ── 5. CHUẨN HÓA BRAND ───────────────────────────────────────
# Lỗi: "Làng Chài Xưa" vs "Làng chài xưa" (khác chữ hoa)
# Lỗi: "-" = không xác định brand

df['brand'] = df['brand'].str.strip()

brand_map = {
    'Làng chài xưa': 'Làng Chài Xưa',
    '-': 'Unknown',
    'THUẬN PHÁT HƯNG': 'THUẬN PHÁT',   # gộp về 1 brand
}
df['brand'] = df['brand'].replace(brand_map)

# Chuẩn hóa toàn bộ: strip khoảng trắng thừa
df['brand'] = df['brand'].str.strip()

print("\n✅ Danh sách brand sau khi chuẩn hóa:")
print(sorted(df['brand'].unique()))


# ── 6. TẠO CỘT PHÂN TÍCH BỔ SUNG ────────────────────────────

# Phân khúc giá
def price_segment(price):
    if pd.isna(price):
        return 'Unknown'
    elif price < 50000:
        return 'Rẻ (<50k)'
    elif price <= 150000:
        return 'Trung (50-150k)'
    else:
        return 'Cao (>150k)'

df['phan_khuc_gia'] = df['gia_ban'].apply(price_segment)

# % doanh số theo brand (so với tổng 200 sản phẩm)
tong_ds = df['doanh_so_365'].sum()
df['pct_doanh_so'] = (df['doanh_so_365'] / tong_ds * 100).round(2)

# Tỉ lệ đánh giá trên lượt bán (proxy chất lượng)
df['ti_le_danh_gia'] = (df['tong_danh_gia'] / df['tong_luot_ban']).round(4)

print("\n✅ Các cột mới đã tạo:")
print(df[['ten_sp', 'phan_khuc_gia', 'pct_doanh_so', 'ti_le_danh_gia']].head())


# ── 7. KIỂM TRA NULL ─────────────────────────────────────────
print("\n✅ Kiểm tra null:")
print(df.isnull().sum())


# ── 8. KIỂM TRA DUPLICATE ────────────────────────────────────
dupes = df[df.duplicated(subset='ten_sp', keep=False)]
print(f"\n✅ Số dòng trùng tên sản phẩm: {len(dupes)}")
if len(dupes) > 0:
    print(dupes[['stt', 'ten_sp', 'brand']])


# ── 9. EXPORT FILE SẠCH ──────────────────────────────────────
df.to_csv('clean_dataset.csv', index=False, encoding='utf-8-sig')
# utf-8-sig để mở đúng tiếng Việt trong Excel và Looker Studio
print("\n🎉 Done! File đã lưu: clean_dataset.csv")
print("Shape cuối:", df.shape)

Shape gốc: (200, 9)
   STT                                       Tên sản phẩm  Giá bán  \
0    1  [Tặng hộp hoặc sốt] Combo 3 gói sốt muối kim c...   91.100   
1    2  Sa Tế Sò Điệp Thích Cay Sốt Trộn Mì, Hủ Tiếu, ...   59.000   
2    3  Nước Mắm Tĩn Nhãn Đỏ Độ Đạm 40N Cặp 2 Chai Thủ...  330.000   

  Doanh số 365 ngày gần nhất  Sản phẩm đã bán trong khoảng 365 ngày gần nhất  \
0              2.383.563.000                                          24.148   
1                812.970.720                                          14.151   
2                679.857.000                                           2.073   

   Tổng doanh số  Tổng lượt bán  Tổng đánh giá         Brand  
0  4.999.687.040         50.686         11.674        O'food  
1  4.396.505.105         78.015         17.262     THÍCH CAY  
2  1.181.512.000          3.601        746.000  Nước Mắm Tĩn  

✅ Giá bán sau khi clean (5 dòng đầu):
0     91100
1     59000
2    330000
3     63000
4    145000
Name: gia_ban, dtype: int64

In [ ]:
# ============================================================
# BƯỚC 2: PHÂN TÍCH CƠ BẢN — Top 200 Gia Vị Shopee
# Chạy tiếp sau buoc1_data_cleaning.py
# ============================================================

import pandas as pd
import numpy as np

# ── LOAD DATA SẠCH ───────────────────────────────────────────
df = pd.read_csv('clean_dataset.csv')
print("✅ Loaded:", df.shape)


# ══════════════════════════════════════════════════════════════
# PHÂN TÍCH 1: THỊ PHẦN THEO BRAND
# ══════════════════════════════════════════════════════════════
brand_summary = df.groupby('brand').agg(
    so_SKU        = ('ten_sp', 'count'),
    tong_ds_365   = ('doanh_so_365', 'sum'),
    tong_luot_ban = ('tong_luot_ban', 'sum'),
    ds_tb_per_SKU = ('doanh_so_365', 'mean'),
    gia_tb        = ('gia_ban', 'mean'),
).reset_index()

brand_summary['pct_doanh_so'] = (
    brand_summary['tong_ds_365'] / brand_summary['tong_ds_365'].sum() * 100
).round(2)

brand_summary = brand_summary.sort_values('tong_ds_365', ascending=False)

print("\n📊 THỊ PHẦN THEO BRAND (Top 10):")
print(brand_summary.head(10).to_string(index=False))


# ══════════════════════════════════════════════════════════════
# PHÂN TÍCH 2: PHÂN KHÚC GIÁ
# ══════════════════════════════════════════════════════════════
price_seg = df.groupby('phan_khuc_gia').agg(
    so_san_pham   = ('ten_sp', 'count'),
    tong_ds       = ('doanh_so_365', 'sum'),
    tong_luot_ban = ('tong_luot_ban', 'sum'),
).reset_index()

price_seg['pct_doanh_so'] = (
    price_seg['tong_ds'] / price_seg['tong_ds'].sum() * 100
).round(2)

price_seg = price_seg.sort_values('tong_ds', ascending=False)

print("\n📊 PHÂN KHÚC GIÁ:")
print(price_seg.to_string(index=False))


# ══════════════════════════════════════════════════════════════
# PHÂN TÍCH 3: PHÂN LOẠI SẢN PHẨM (category tự gán)
# ══════════════════════════════════════════════════════════════
# Gán category dựa trên keyword trong tên sản phẩm
def categorize(name):
    name = name.lower()
    if any(k in name for k in ['nước mắm', 'mắm nêm', 'mắm tôm']):
        return 'Nước mắm & Mắm'
    elif any(k in name for k in ['sa tế', 'sốt', 'xốt', 'tương ớt', 'tương cà']):
        return 'Sốt & Gia vị pha sẵn'
    elif any(k in name for k in ['muối', 'bột', 'gia vị', 'hạt nêm']):
        return 'Muối & Bột gia vị'
    elif any(k in name for k in ['nước tương', 'xì dầu', 'dầu hào']):
        return 'Nước tương & Dầu hào'
    elif any(k in name for k in ['mì', 'canh', 'lẩu', 'phở', 'bún']):
        return 'Gia vị nấu & Mì ăn liền'
    elif any(k in name for k in ['dầu ăn', 'dầu màu']):
        return 'Dầu ăn'
    else:
        return 'Khác'

df['category'] = df['ten_sp'].apply(categorize)

cat_summary = df.groupby('category').agg(
    so_san_pham = ('ten_sp', 'count'),
    tong_ds     = ('doanh_so_365', 'sum'),
    gia_tb      = ('gia_ban', 'mean'),
).reset_index()

cat_summary['pct_doanh_so'] = (
    cat_summary['tong_ds'] / cat_summary['tong_ds'].sum() * 100
).round(2)

cat_summary = cat_summary.sort_values('tong_ds', ascending=False)

print("\n📊 PHÂN LOẠI SẢN PHẨM:")
print(cat_summary.to_string(index=False))


# ══════════════════════════════════════════════════════════════
# PHÂN TÍCH 4: TOP 10 SẢN PHẨM BÁN CHẠY NHẤT
# ══════════════════════════════════════════════════════════════
top10 = df.nlargest(10, 'doanh_so_365')[
    ['stt', 'ten_sp', 'brand', 'gia_ban', 'doanh_so_365',
     'so_luong_ban_365', 'tong_danh_gia', 'phan_khuc_gia']
]
print("\n📊 TOP 10 SẢN PHẨM BÁN CHẠY NHẤT:")
print(top10.to_string(index=False))


# ══════════════════════════════════════════════════════════════
# PHÂN TÍCH 5: WHITE SPACE — Cơ hội thị trường
# Doanh số cao nhưng đánh giá thấp → ít cạnh tranh chất lượng
# ══════════════════════════════════════════════════════════════
median_ds = df['doanh_so_365'].median()
median_review = df['ti_le_danh_gia'].median()

df['co_hoi'] = (
    (df['doanh_so_365'] > median_ds) &
    (df['ti_le_danh_gia'] < median_review)
)

white_space = df[df['co_hoi']][
    ['ten_sp', 'brand', 'gia_ban', 'doanh_so_365', 'ti_le_danh_gia', 'category']
].sort_values('doanh_so_365', ascending=False)

print(f"\n📊 WHITE SPACE — Sản phẩm doanh số cao, ít review ({len(white_space)} SP):")
print(white_space.head(10).to_string(index=False))


# ══════════════════════════════════════════════════════════════
# EXPORT — lưu toàn bộ kết quả ra Excel nhiều sheet
# ══════════════════════════════════════════════════════════════

# Thêm cột category vào df chính và lưu lại
df.to_csv('clean_dataset.csv', index=False, encoding='utf-8-sig')

with pd.ExcelWriter('analysis_results.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Full Data', index=False)
    brand_summary.to_excel(writer, sheet_name='Brand Summary', index=False)
    price_seg.to_excel(writer, sheet_name='Price Segment', index=False)
    cat_summary.to_excel(writer, sheet_name='Category', index=False)
    top10.to_excel(writer, sheet_name='Top 10 SP', index=False)
    white_space.to_excel(writer, sheet_name='White Space', index=False)

print("\n🎉 Done! File đã lưu: analysis_results.xlsx")
print("   → 6 sheets: Full Data, Brand Summary, Price Segment, Category, Top 10 SP, White Space")

✅ Loaded: (200, 12)

📊 THỊ PHẦN THEO BRAND (Top 10):
        brand  so_SKU  tong_ds_365  tong_luot_ban  ds_tb_per_SKU        gia_tb  pct_doanh_so
     Cholimex      16 158750968576         114014   9.921936e+09  33453.750000         62.32
Làng Chài Xưa      26  22774403775         127416   8.759386e+08 173865.384615          8.94
     Dh Foods      21  15553732479         185252   7.406539e+08  29600.761905          6.11
       O'food      21  14008427753         259531   6.670680e+08 105504.761905          5.50
    THÍCH CAY      25  12382459917         180867   4.952984e+08  90736.000000          4.86
       Ottogi      21  10759027724         144936   5.123347e+08  99313.619048          4.22
 Nước Mắm Tĩn      17   8361151015          58738   4.918324e+08 437705.882353          3.28
       Barona      14   6836084778          78340   4.882918e+08 119456.071429          2.68
   THUẬN PHÁT       8   1257565414          26735   1.571957e+08  94500.000000          0.49
       I-soup    